In [1]:
import pandas as pd
import jdatetime as jdt
import jalali_pandas

In [ ]:
woodinc = pd.read_excel(r"D:\تحلیل داده\python\تمرین\WoodInc-sale.xlsx")
woodinc

In [ ]:
def to_jalali_date(column, year_dir='L'):
    column = pd.Series(column)

    if column.dtype == 'object':
        sep = next((s for s in ['/', '-', '.'] if s in str(column.iloc[0])), None)
        mod_col = [x.split(sep) for x in column]
    
    elif pd.api.types.is_integer_dtype(column):
        str_col = column.astype(str)
        sample = str_col.iloc[0]
        if len(sample) == 8:
            if year_dir == 'L':
                mod_col = [[x[:4], x[4:6], x[6:]] for x in str_col]
            else:
                mod_col = [[x[:2], x[2:4], x[4:]] for x in str_col]
        elif len(sample) == 6:
            mod_col = [[x[:2], x[2:4], x[4:]] for x in str_col]
        else:
            raise ValueError("Unsupported integer date format")

    else:
        raise TypeError("Unsupported column dtype")

    # Extract year, month, day based on year direction
    idx = (0, 1, 2) if year_dir == 'L' else (2, 1, 0)
    year, month, day = zip(*[(i[idx[0]], i[idx[1]], i[idx[2]]) for i in mod_col])

    # Fix 2-digit years
    year = [
        f"14{y}" if len(y) == 2 and int(y) < 50 else
        f"13{y}" if len(y) == 2 and int(y) >= 50 else
        y
        for y in year
    ]

    return [jdt.date(int(y), int(m), int(d)) for y, m, d in zip(year, month, day)]
    woodinc['تاريخ'] = to_jalali_date(woodinc['تاريخ'])
woodinc

,تاريخ,شماره,کد مشتري,نام مشتري,کد کالا,نام کالا,واحد,مقدار,في,مبلغ,ماليات,مبلغ خالص
0,1399-01-05,82516,1101328,ابوالفضل تام ...,1-1-04-706,ملامينه 210*280 كد 361 سفيد صابوني كيميا ...,ورق ...,5.0,2339450,11697250,1052753,12750003
1,1399-01-05,82516,1101328,ابوالفضل تام ...,1-1-01-505,اچ دي اف 3 ميل 210*244 سفيد پرينتي درجه A ...,ورق ...,2.0,596330,1192660,107339,1299999
2,1399-01-05,82516,1101328,ابوالفضل تام ...,1-1-04-023,ملامينه 204*260 كد 79 آنتيك سويز درجه A ...,ورق ...,5.0,2293578,11467890,1032110,12500000
3,1399-01-05,82517,1100140,احساني عليرضا ...,1-1-05-137,نئوپان 210*366 خام ممتاز گلستان درجه B ...,ورق ...,3.0,1788991,5366973,483028,5850001
4,1399-01-05,82518,1101464,مطهري مهدي ...,1-1-01-294,ام دي اف 183*366 سفيد صابوني كيميا چوب ...,ورق ...,2.0,3669725,7339450,660551,8000001
...,...,...,...,...,...,...,...,...,...,...,...,...
28269,1399-12-24,98776,1100005,سبزي مهدي ...,1-1-05-056,نئوپان 210*305 خام درجه 1 ...,ورق ...,70.0,3348624,234403680,21096331,255500011
28270,1399-12-25,98826,1102206,محمد جواد حيدري ...,1-1-01-029,ام دي اف 210*366 كد 60 ونگه درجه A ...,ورق ...,2.0,10825688,21651376,1948624,23600000
28271,1399-12-26,98871,1100140,احساني عليرضا ...,1-1-04-157,ملامينه 210*366 كد 70 گردويي تيره درجه A ...,ورق ...,2.0,8440367,16880734,1519266,18400000
28272,1399-12-27,98948,1101709,سيد علي موسوي ...,1-1-05-046,نئوپان 204*260 خام 2% ...,ورق ...,10.0,2935780,29357800,2642202,32000002


In [ ]:
woodinc['month'] = woodinc['تاريخ'].jalali.month
woodinc.groupby('month').agg(sumsale = ('مبلغ خالص','sum') , avgpricesale= ('في','mean'),countname = ('نام مشتري','count')).reset_index()

,month,sumsale,avgpricesale,countname
0,1,21789957494,1.958121e+06,1221
1,2,36217161779,2.006550e+06,2366
2,3,30643889582,2.285392e+06,2113
3,4,64222149991,2.691524e+06,3212
4,5,66578272163,3.196476e+06,2846
5,6,59029056413,2.816206e+06,2609
6,7,74952726066,3.404953e+06,2605
7,8,56156461774,4.061874e+06,2109
8,9,73790978679,4.691327e+06,2044
9,10,91561359520,4.386980e+06,2488


In [ ]:
woodinc['نام مشتري']=woodinc['نام مشتري'].str.strip()
group1 = woodinc.groupby(['month','نام مشتري']).agg(sumsale = ('مبلغ خالص','sum')).reset_index()
group1

,month,نام مشتري,sumsale
0,1,آسيابان سعيد,91780011
1,1,آقا زيارتي علي,798960012
2,1,آقاي ارديبهشت,374220061
3,1,آقاي وحيدي,40709997
4,1,ابرقوئي وحيد,296810004
...,...,...,...
2347,12,وحيدي اسد,157099989
2348,12,يعقوبي سپهر,65799998
2349,12,پور بافراني نعمت,48749997
2350,12,پيام كارگشا,61410002


In [ ]:
x=(group1.sort_values(['sumsale','month'],ascending=False).groupby('month').head(n=1))
x


,month,نام مشتري,sumsale
1972,11,احسان مولوي,7734060200
1712,9,مصطفي نوحي,7587810086
1529,8,مصطفي نوحي,7479590085
2279,12,فخار زاده هادي,6845919869
1370,7,مصطفي نوحي,6060040096
1755,10,احسان مولوي,5748889903
1120,6,فخار زاده هادي,4181290029
147,2,حاج ابوئي,3822611987
527,4,توسلي,3664100127
895,5,غلامي محمد,3656555038
